In [ ]:
# Scenario 1: Operational Baseline Analysis
**IoT Network Forensics Master's Thesis**

The extracted Zeek metadata is then ingested into a Python data analysis environment utilizing the Pandas library. In this phase, the raw datasets are cleansed of environmental background noise, such as standard Mininet overhead and Layer 2 broadcast traffic. The dataset is filtered to isolate the flows specifically associated with the target IoT IP addresses and the gateway. Finally, critical behavioral metrics are mathematically derived from the timestamps, most notably the Inter-Arrival Time (IAT) between specific device communications.

**Topology Definition:**
* Smart Camera: `10.0.0.1`
* Smart Thermostat: `10.0.0.2`
* Attacker Node: `10.0.0.3`
* Cloud/Gateway: `10.0.0.254`
* Duration: 1200 seconds (20 minutes)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from zat.log_to_dataframe import LogToDataFrame

# Set academic plotting style for LaTeX embedding
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "axes.labelsize": 12,
    "font.size": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300
})

# Define paths
LOG_DIR = "../../zeek_logs/scenario1_logs/"  # Adjust relative path if needed
OUTPUT_DIR = "../../figures/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Data Ingestion & Preprocessing
We utilize the Zeek Analysis Tools (`zat`) library to safely parse the TSV logs, automatically handling the commented headers and converting Zeek's epoch timestamps into native Python Datetime indices. We then filter out any non-IoT subnet noise (e.g., Mininet IPv6/multicast background traffic).

In [ ]:
# Initialize ZAT parser
log_to_df = LogToDataFrame()

# Ingest Zeek Logs
conn_df = log_to_df.create_dataframe(os.path.join(LOG_DIR, 'conn.log'))
dns_df = log_to_df.create_dataframe(os.path.join(LOG_DIR, 'dns.log'))

# Preprocessing: Ensure timestamps are timezone-naive for easier math
if conn_df.index.tz is not None:
    conn_df.index = conn_df.index.tz_localize(None)
if dns_df.index.tz is not None:
    dns_df.index = dns_df.index.tz_localize(None)

# Define our forensic scope
target_ips = ['10.0.0.1', '10.0.0.2', '10.0.0.3', '10.0.0.254']

# Filter conn.log to only include traffic where our topology nodes are the originators
conn_df = conn_df[conn_df['id.orig_h'].isin(target_ips)]
dns_df = dns_df[dns_df['id.orig_h'].isin(target_ips)]

print(f"Total connections ingested: {len(conn_df)}")
print(f"Total DNS queries ingested: {len(dns_df)}")